# Problem 029:  Distinct Powers

> Consider all integer combinations of $a^b$ for $2 \le a \le 5$ and $2 \le b \le 5$:
> \begin{array}{rrrr}
2^2=4, &2^3=8, &2^4=16, &2^5=32\\
3^2=9, &3^3=27, &3^4=81, &3^5=243\\
4^2=16, &4^3=64, &4^4=256, &4^5=1024\\
5^2=25, &5^3=125, &5^4=625, &5^5=3125
\end{array}
>
> If they are then placed in numerical order, with any repeats removed, we get the following sequence of $15$ distinct terms:
> $$4, 8, 9, 16, 25, 27, 32, 64, 81, 125, 243, 256, 625, 1024, 3125.$$
>
> How many distinct terms are in the sequence generated by $a^b$ for $2 \le a \le 100$ and $2 \le b \le 100$?

## Naive solution $\mathcal O\left(N M\right)$

Here's a one-liner in Python.

Create a list of all the numbers, regardless of the size it may entail.  Cast that list into a set to remove duplicates.  Return the length of that set.

Since you are calculating all the values on the $N$ by $M$ grid, this approach has a time complexity of $\mathcal O(NM)$.

In [2]:
%%time

def p029_naive(N: int, M: int) -> int:
    return len(set([pow(a,b) for a in range(2,N+1) for b in range(2,M+1)]))

N = 100
M = 100
ans = p029_naive(N, M)

print(ans)

9183
CPU times: user 8.35 ms, sys: 1.92 ms, total: 10.3 ms
Wall time: 9.74 ms


## A faster approach $\mathcal O\left(N\ln N\right)$

The inefficiency of the above one-liner is glaring.  You need to calculate overly large values like $N^M$, and the quadratic time scaling leaves something to be desired.

Let's consider where the duplicated elements are coming from.  Namely, they come from when two bases are related to each other by a power, like $2$ and $4$; all the even powers from the row with base $2$ will be duplicated in the base $4$ row, with the excpetion of $2^2$.  So, let's consider the powers observed in the table for each base that is itself not a power.  Furthermore, the identity of the base $a$ is irrelevent and the only thing we need to worry about is the maximum power $c$ s.t. $a^c <= N$. Then, given this maximum power $c$, count how many unique powers of that base are observed in the whole table.

First, find the values $a<=N$ that cannot be written as a power of a smaller number.  Calculate the maximum power $c$ for eahc of these $a$'s and make a histogram as a function of $c$.  To do this, I create a sieve that is initiated to `True` for all values up to and including $N$.  Then I increment through the sieve starting at $a=2$.  If the sieve is `True`, then that number cannot be written as a power of a smaller number.  Then I multiply $a$ with itself until I get a value greater than $N$, switching the sieve to `False` for each value I go through.  This process thus counts up to the maximum exponent $c$ for the base and updates the sieve to remove any values that can be written as a power of a smaller number.  This process has a time complexity of $\mathcal O(N)$, since $\sum_n \ln(N) / \ln(n) \sim N$.

Second, given the maximum exponent $c$ for each value below $N$, the number of powers in the table for the base needs to be calculated.  The numbers that are considered are $\{2-M\}$ for $a^1$, all the even numbers from $\{2-2M\}$ for $a^2$, all the divisible by $3$ numbers between $\{2-3M\}$ for $a^3$, and so on up to all the divisible by $c$ numbers between $\{2-cM\}$ for $a^c$.  PIE can be used to find the total number of values satisfying these conditions without overcounting.  To this end, the number of values that satisfy being between $\{2-xM\}$ and divisible by $x$ for all values $x$ in a set $X$ will be $\left\lfloor \frac{M \min(\{X\})}{\mbox{lcm}(\{X\})}\right\rfloor$.  Better yet, PIE can be done such that the value for smaller $c$ can be used for $c+1$, avoiding duplicate calculations.  The time complexity of this will be $2^{c_{max}}c_{max}\ln c_{max}$ since PIE requires $2^c$ calculations and the LCM calculation is approximately of order $c\ln c$.  Noticing that $N \simeq 2^{c_{max}}$, the time complexity will be $N\ln N$.

This approach is obviously more complicated than the previous approach, but it allows for calculating much larger values.  For example, the first approach would need to make $10^{12}$ calculations, some with $6,000,000$ digits, for a $10^6$ by $10^6$ table.  The code below calculates the value within $2$ seconds on my machine.

In [5]:
%%time

from itertools import combinations
from math import lcm

def p029(N: int, M: int) -> int:
    val = 0

    s = [True] * (N+1)
    jmax = 0
    j = N // 4
    while j:
        j //= 2
        jmax += 1
    power = [0] * (jmax+1)
    
    for i in range(2, N+1):
        if s[i]:
            j = 0
            k = i
            while k <= N:
                s[k] = False
                k *= i
                j += 1
            power[j-1] += 1
    
    cnt = [0] * (jmax+1)
    val = M - 1
    cnt[0] = val
    for j in range(2, jmax + 2):
        parity = 1
        for n in range(j):
            for x in combinations(range(1, j), n):
                if x:
                    xmin = x[0]
                else:
                    xmin = j
                #print(x, j, lcm(*x,j), xmin, parity * ((M * xmin) // lcm(*x,j)))
                val += parity * ((M * xmin) // lcm(*x,j))
            parity = -parity
        cnt[j-1] = val
        #print()
    #print(cnt)
    val = 0
    for i in range(jmax+1):
        val += cnt[i] * power[i]
        #print(i,cnt[i], power[i])
    return val

N = 100
M = 100
ans = p029(N, M)

print(ans)

9183
CPU times: user 251 μs, sys: 41 μs, total: 292 μs
Wall time: 282 μs
